# 02 - Captura y Exploración de Datos

Este notebook explora los screenshots capturados de Google Maps con la capa de tráfico.
Muestra cómo funciona la captura, ejemplos de imágenes y estadísticas del dataset.

**Pipeline:**
1. Configuración de la captura (área, período, zoom)
2. Exploración de screenshots capturados
3. Ejemplo de generación de HTML con capa de tráfico
4. Estadísticas del dataset de imágenes

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import re
from datetime import datetime
from collections import Counter

## 1. Configuración de la captura

In [2]:
from src.capture.config import CONFIG

print("Configuración de captura:")
print(f"  API Key: {CONFIG['API_KEY'][:10]}...")
print(f"  Coordenadas:")
print(f"    SW: {CONFIG['COORDS']['southwest']}")
print(f"    NE: {CONFIG['COORDS']['northeast']}")
print(f"  Zoom: {CONFIG['ZOOM']}")
print(f"  Período: {CONFIG['PERIOD_MINUTES']} minutos")

ModuleNotFoundError: No module named 'src'

## 2. Exploración de screenshots capturados

In [3]:
SCREENSHOTS_DIR = 'data/raw/Images/screenshotsGoogleMaps/screenshots'

# Listar todas las imágenes PNG
png_files = sorted([f for f in os.listdir(SCREENSHOTS_DIR) if f.endswith('.png')])
print(f"Total de screenshots: {len(png_files)}")
print(f"Primer archivo: {png_files[0]}")
print(f"Último archivo: {png_files[-1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/Images/screenshotsGoogleMaps/screenshots'

In [ ]:
# Analizar distribución temporal
dates = []
hours = []
weekdays = []

for f in png_files:
    match = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})', f)
    if match:
        year, month, day, hour, minute = match.groups()
        dt = datetime(int(year), int(month), int(day))
        dates.append(dt.date())
        hours.append(int(hour))
        weekdays.append(dt.strftime('%A'))

print(f"Período: {min(dates)} a {max(dates)}")
print(f"Días únicos: {len(set(dates))}")
print(f"Horas representadas: {sorted(set(hours))}")

In [ ]:
# Distribución por día de la semana
weekday_counts = Counter(weekdays)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Screenshots por día
date_counts = Counter(dates)
axes[0].bar(range(len(date_counts)), list(date_counts.values()))
axes[0].set_xticks(range(len(date_counts)))
axes[0].set_xticklabels([str(d) for d in date_counts.keys()], rotation=45, fontsize=8)
axes[0].set_ylabel('Número de screenshots')
axes[0].set_title('Screenshots por día')

# Distribución por hora
axes[1].hist(hours, bins=24, range=(0, 24), edgecolor='black')
axes[1].set_xlabel('Hora del día')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución por hora')
axes[1].set_xticks(range(0, 25, 3))

plt.tight_layout()
plt.savefig('results/figures/distribucion_temporal_screenshots.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Ejemplo de imágenes de tráfico

In [ ]:
# Mostrar screenshots de diferentes horas del mismo día
ejemplo_dia = '2023-04-03'  # Lunes
ejemplos = [f for f in png_files if ejemplo_dia in f][:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, filename in zip(axes.flat, ejemplos):
    img = cv2.imread(os.path.join(SCREENSHOTS_DIR, filename))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    hora = re.search(r'_(\d{2})-(\d{2})', filename).group(0)
    ax.set_title(f'{hora}', fontsize=12)
    ax.axis('off')

plt.suptitle(f'Evolución del tráfico - {ejemplo_dia}', fontsize=16)
plt.tight_layout()
plt.savefig('results/figures/evolucion_trafico_ejemplo.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Análisis de colores de tráfico

In [ ]:
# Definición de colores de tráfico de Google Maps
TRAFFIC_COLORS = {
    'green':    {'rgb': (76, 175, 80),   'valor': 64,  'label': 'Fluido'},
    'orange':   {'rgb': (255, 152, 0),   'valor': 128, 'label': 'Moderado'},
    'red':      {'rgb': (242, 60, 50),   'valor': 191, 'label': 'Lento'},
    'dark_red': {'rgb': (129, 31, 31),   'valor': 255, 'label': 'Muy lento'},
}

# Visualizar escala de colores
fig, ax = plt.subplots(figsize=(10, 2))
colors_list = [v['rgb'] for v in TRAFFIC_COLORS.values()]
labels_list = [f"{v['label']}\n({v['valor']})" for v in TRAFFIC_COLORS.values()]

for i, (color, label) in enumerate(zip(colors_list, labels_list)):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=[c/255 for c in color]))
    ax.text(i + 0.5, 0.5, label, ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xlim(0, 4)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Escala de colores de tráfico de Google Maps', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/escala_colores_trafico.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Ejemplo de conversión de imagen a matriz de tráfico
ejemplo_file = os.path.join(SCREENSHOTS_DIR, png_files[100])
img = cv2.imread(ejemplo_file)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(rgb)
axes[0].set_title('Imagen original')

# Detectar verde (fluido)
mask_green = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
axes[1].imshow(mask_green, cmap='Greens')
axes[1].set_title('Máscara verde (fluido)')

# Detectar naranja (moderado)
mask_orange = cv2.inRange(hsv, np.array([10, 100, 100]), np.array([25, 255, 255]))
axes[2].imshow(mask_orange, cmap='Oranges')
axes[2].set_title('Máscara naranja (moderado)')

for ax in axes:
    ax.axis('off')

plt.suptitle(f'Detección de colores - {os.path.basename(ejemplo_file)}', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/deteccion_colores_ejemplo.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Generación del HTML de captura (referencia)

In [ ]:
from src.capture.map_generator import generate_map_html

# Generar HTML de ejemplo (sin ejecutar la captura)
output_html = 'map_ejemplo.html'
generate_map_html(output_html)
print(f"HTML generado: {output_html}")
print("Para capturar screenshots, ejecutar: python src/run_capture.py")

## 6. Resumen del dataset

In [ ]:
# Tamaño promedio de las imágenes
sample_img = cv2.imread(os.path.join(SCREENSHOTS_DIR, png_files[0]))
h, w = sample_img.shape[:2]

print("=" * 50)
print("RESUMEN DEL DATASET DE IMÁGENES")
print("=" * 50)
print(f"Total de imágenes: {len(png_files)}")
print(f"Dimensiones: {w} x {h} píxeles")
print(f"Período: {min(dates)} a {max(dates)}")
print(f"Días capturados: {len(set(dates))}")
print(f"Frecuencia: cada {CONFIG['PERIOD_MINUTES']} minutos")
print(f"Área: Punta Arenas, Chile")
print(f"Zoom: {CONFIG['ZOOM']}")
print("=" * 50)